# Lab 07.2: Bronze → Silver - Datos Curados y Modelo Dimensional
## TransCore Data Engineer - Módulo 07

**Objetivo**: Cargar datos desde raw/, limpiar, curar y crear modelo dimensional star schema.

**Input**: `s3://transcore-infra-prod-eu-west-1-ric/raw/obra-lineal/año=2026/mes=05/dia=06/partes_trabajo_2026-05-06.csv`

**Output**: Star schema en Parquet en `s3://transcore-infra-prod-eu-west-1-ric/silver/obra-lineal/año=2026/mes=05/dia=06/`
- `fact_ordenes_2026-05-06.parquet`
- `dim_tiempo_2026-05-06.parquet`
- `dim_tipo_mantenimiento_2026-05-06.parquet`
- `dim_activos_2026-05-06.parquet`

## 1. Configuración AWS

In [ ]:
# Configuración de credenciales AWS (S3 real)
import os

os.environ["AWS_ACCESS_KEY_ID"] = "<TU_ACCESS_KEY_ID>"
os.environ["AWS_SECRET_ACCESS_KEY"] = "<TU_SECRET_ACCESS_KEY>"


# Limpiar cualquier configuración de LocalStack
os.environ.pop("AWS_ENDPOINT_URL", None)
os.environ.pop("AWS_ENDPOINT", None)

# Configurar región de AWS explícitamente para AWS real
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

print("Configuración: AWS real, región eu-west-1")

## 2. Importar Libraries

In [ ]:
import numpy as np
import pandas as pd
import boto3
from datetime import datetime
from pathlib import Path
import tempfile

print("Librerías importadas correctamente")

## 3. Configuración de S3 y Paths

In [ ]:
# Configuración S3
BUCKET_NAME = "<TU_BUCKET_NAME>"
FECHA_PROCESO = "2026-05-06"

# Paths
RAW_KEY = f"raw/obra-lineal/año=2026/mes=05/dia=06/partes_trabajo_{FECHA_PROCESO}.csv"
SILVER_PREFIX = f"silver/obra-lineal/año=2026/mes=05/dia=06"

# Crear cliente S3
s3_client = boto3.client("s3", region_name="eu-west-1")

print(f"Bucket: {BUCKET_NAME}")
print(f"Raw key: {RAW_KEY}")
print(f"Silver prefix: {SILVER_PREFIX}")

## PARTE A: LIMPIEZA Y CURACIÓN DE DATOS

## 4. Cargar Datos desde Raw/

In [ ]:
def load_csv_from_s3(bucket, key):
    """Carga un CSV directamente desde S3 a un DataFrame."""
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(obj['Body'])

# Cargar dataset desde S3 raw
print("Cargando datos desde raw...")
ordenes = load_csv_from_s3(BUCKET_NAME, RAW_KEY)
print(f"✓ Órdenes cargadas: {len(ordenes):,} registros")
print(f"  Columnas: {ordenes.columns.tolist()}")

## 5. Convertir Tipos de Datos

In [ ]:
# Convertir columnas de fecha a datetime
ordenes['fecha_inicio'] = pd.to_datetime(ordenes['fecha_inicio'])
ordenes['fecha_fin'] = pd.to_datetime(ordenes['fecha_fin'])

# Convertir horas_trabajo a float
ordenes['horas_trabajo'] = pd.to_numeric(ordenes['horas_trabajo'], errors='coerce')

print("=== Tipos de Datos Convertidos ===")
print(f"  - fecha_inicio: {ordenes['fecha_inicio'].dtype}")
print(f"  - fecha_fin: {ordenes['fecha_fin'].dtype}")
print(f"  - horas_trabajo: {ordenes['horas_trabajo'].dtype}")
print(f"\nRango de fechas:")
print(f"  - Inicio: {ordenes['fecha_inicio'].min()} a {ordenes['fecha_inicio'].max()}")
print(f"  - Fin: {ordenes['fecha_fin'].min()} a {ordenes['fecha_fin'].max()}")

## 6. Validaciones de Calidad y Limpieza

In [ ]:
registros_iniciales = len(ordenes)
print(f"=== Iniciando Limpieza ===")
print(f"Registros iniciales: {registros_iniciales:,}")

# 1. Eliminar duplicados por parte_id
registros_antes = len(ordenes)
ordenes = ordenes.drop_duplicates(subset=['parte_id'])
duplicados = registros_antes - len(ordenes)
print(f"\n✓ Duplicados eliminados: {duplicados:,}")

# 2. Filtrar horas_trabajo inválidas
registros_antes = len(ordenes)
ordenes = ordenes[
    (ordenes['horas_trabajo'] >= 0) & 
    (ordenes['horas_trabajo'] <= 200)
]
invalidos_horas = registros_antes - len(ordenes)
print(f"✓ Registros con horas_trabajo inválidas eliminados: {invalidos_horas:,}")

# 3. Filtrar órdenes donde fecha_inicio > fecha_fin
registros_antes = len(ordenes)
ordenes = ordenes[
    (ordenes['fecha_fin'].isna()) | 
    (ordenes['fecha_inicio'] <= ordenes['fecha_fin'])
]
invalidos_fechas = registros_antes - len(ordenes)
print(f"✓ Órdenes con fecha_inicio > fecha_fin eliminados: {invalidos_fechas:,}")

# 4. Validar que equipo_id no sea nulo
registros_antes = len(ordenes)
ordenes = ordenes[ordenes['equipo_id'].notna()]
sin_equipo = registros_antes - len(ordenes)
print(f"✓ Registros sin equipo_id eliminados: {sin_equipo:,}")

# 5. Validar tipo_mantenimiento
tipos_validos = ['PREVENTIVO', 'CORRECTIVO', 'PREDICTIVO', 'EMERGENCIA', 'INSPECCION']
registros_antes = len(ordenes)
ordenes = ordenes[ordenes['tipo_mantenimiento'].isin(tipos_validos)]
tipos_invalidos = registros_antes - len(ordenes)
print(f"✓ Registros con tipo_mantenimiento inválido eliminados: {tipos_invalidos:,}")

print(f"\n=== Resumen de Limpieza ===")
print(f"Registros iniciales: {registros_iniciales:,}")
print(f"Registros finales: {len(ordenes):,}")
print(f"Registros eliminados: {registros_iniciales - len(ordenes):,} ({(registros_iniciales - len(ordenes))/registros_iniciales*100:.2f}%)")

## 7. Calcular Columnas Derivadas

In [ ]:
# Calcular duración en horas de cada orden de mantenimiento
ordenes['duracion_horas'] = (
    (ordenes['fecha_fin'] - ordenes['fecha_inicio'])
    .dt.total_seconds() / 3600
)

print("=== Columna Derivada: duracion_horas ===")
print(f"Media de duración: {ordenes['duracion_horas'].mean():.2f} horas")
print(f"Mediana de duración: {ordenes['duracion_horas'].median():.2f} horas")
print(f"Min: {ordenes['duracion_horas'].min():.2f} horas")
print(f"Max: {ordenes['duracion_horas'].max():.2f} horas")
print(f"Órdenes sin fecha_fin (NaN): {ordenes['fecha_fin'].isna().sum()}")

## 8. Añadir Metadata

In [ ]:
# Añadir metadata de procesamiento
ordenes['_procesado_timestamp'] = datetime.now().isoformat()

print("✓ Metadata de procesamiento añadida")

## PARTE B: MODELO DIMENSIONAL

## 9. Crear Dimensión: dim_tiempo

In [ ]:
# Generar rango de fechas único desde fecha_inicio hasta fecha_fin
fechas_unicas = pd.date_range(
    start=ordenes['fecha_inicio'].min(),
    end=ordenes['fecha_fin'].max(),
    freq='D'
)

# Crear dim_tiempo
dim_tiempo = pd.DataFrame({'fecha': fechas_unicas})
dim_tiempo['fecha_id'] = dim_tiempo['fecha'].dt.strftime('%Y%m%d').astype(int)
dim_tiempo['año'] = dim_tiempo['fecha'].dt.year
dim_tiempo['mes'] = dim_tiempo['fecha'].dt.month
dim_tiempo['dia'] = dim_tiempo['fecha'].dt.day
dim_tiempo['dia_semana'] = dim_tiempo['fecha'].dt.dayofweek
dim_tiempo['nombre_dia_semana'] = dim_tiempo['fecha'].dt.day_name()
dim_tiempo['semana_año'] = dim_tiempo['fecha'].dt.isocalendar().week.astype(int)

# Reordenar columnas (fecha_id primero)
dim_tiempo = dim_tiempo[['fecha_id', 'fecha', 'año', 'mes', 'dia', 'dia_semana', 'nombre_dia_semana', 'semana_año']]

print(f"=== dim_tiempo creada ===")
print(f"Total registros: {len(dim_tiempo)}")
print(f"\nPrimeras 5 filas:")
print(dim_tiempo.head())

## 10. Crear Dimensión: dim_tipo_mantenimiento

In [ ]:
# Crear dim_tipo_mantenimiento desde tipos únicos en órdenes
tipos_unicos = ordenes['tipo_mantenimiento'].dropna().unique()

dim_tipo_mant = pd.DataFrame({
    'tipo_mantenimiento': tipos_unicos
})
dim_tipo_mant['tipo_mant_id'] = range(1, len(dim_tipo_mant) + 1)

# Añadir categoría (agrupación de tipos)
categoria_map = {
    'PREVENTIVO': 'Planificado',
    'CORRECTIVO': 'No Planificado',
    'PREDICTIVO': 'Planificado',
    'EMERGENCIA': 'No Planificado',
    'INSPECCION': 'Planificado'
}
dim_tipo_mant['categoria'] = dim_tipo_mant['tipo_mantenimiento'].map(categoria_map)

# Reordenar columnas (tipo_mant_id primero)
dim_tipo_mant = dim_tipo_mant[['tipo_mant_id', 'tipo_mantenimiento', 'categoria']]

print(f"=== dim_tipo_mantenimiento creada ===")
print(f"Total registros: {len(dim_tipo_mant)}")
print(dim_tipo_mant)

## 11. Crear Dimensión: dim_activos

In [ ]:
# Crear dim_activos a partir de los equipos únicos en las órdenes
equipos_unicos = ordenes[['equipo_id']].drop_duplicates()
dim_activos = equipos_unicos.copy()
dim_activos = dim_activos.rename(columns={'equipo_id': 'activo_id'})
dim_activos['nombre_activo'] = 'Equipo_' + dim_activos['activo_id'].astype(str)

# Reordenar columnas (activo_id primero)
dim_activos = dim_activos[['activo_id', 'nombre_activo']]

print(f"=== dim_activos creada ===")
print(f"Total registros: {len(dim_activos)}")
print(f"\nPrimeras 10 filas:")
print(dim_activos.head(10))

## 12. Crear Fact Table: fact_ordenes

In [ ]:
# Crear fact_ordenes con claves foráneas a dimensiones

# Añadir fecha_id basado en fecha_inicio
ordenes_fact = ordenes.copy()
ordenes_fact['fecha_inicio_id'] = ordenes_fact['fecha_inicio'].dt.strftime('%Y%m%d').astype(int)
ordenes_fact['fecha_fin_id'] = ordenes_fact['fecha_fin'].dt.strftime('%Y%m%d').astype('Int64')  # Permite NaN

# Unir con dim_tipo_mant para obtener tipo_mant_id
ordenes_fact = ordenes_fact.merge(
    dim_tipo_mant[['tipo_mant_id', 'tipo_mantenimiento']],
    on='tipo_mantenimiento',
    how='left'
)

# Renombrar equipo_id a activo_id para alinearse con dim_activos
ordenes_fact = ordenes_fact.rename(columns={'equipo_id': 'activo_id', 'parte_id': 'orden_id'})

# Crear fact table con las claves foráneas y métricas
fact_ordenes = ordenes_fact[[
    'orden_id',
    'activo_id',
    'fecha_inicio_id',
    'fecha_fin_id',
    'tipo_mant_id',
    'horas_trabajo',
    'duracion_horas',
    'estado',
    '_procesado_timestamp'
]].copy()

print(f"=== fact_ordenes creada ===")
print(f"Total registros: {len(fact_ordenes)}")
print(f"\nColumnas: {fact_ordenes.columns.tolist()}")
print(f"\nPrimeras 10 filas:")
print(fact_ordenes.head(10))

## 13. Verificar Integridad Referencial

In [ ]:
print("=== Verificando Integridad Referencial ===")

# Verificar que todos los activo_id en fact existen en dim_activos
activos_en_fact = set(fact_ordenes['activo_id'].dropna().unique())
activos_en_dim = set(dim_activos['activo_id'].unique())
activos_huerfanos = activos_en_fact - activos_en_dim

if activos_huerfanos:
    print(f"⚠️  Activos en fact sin dim: {len(activos_huerfanos)}")
else:
    print(f"✓ Todos los activos en fact existen en dim_activos")

# Verificar que todos los tipo_mant_id en fact existen en dim_tipo_mant
tipos_en_fact = set(fact_ordenes['tipo_mant_id'].dropna().unique())
tipos_en_dim = set(dim_tipo_mant['tipo_mant_id'].unique())
tipos_huerfanos = tipos_en_fact - tipos_en_dim

if tipos_huerfanos:
    print(f"⚠️  Tipos en fact sin dim: {len(tipos_huerfanos)}")
else:
    print(f"✓ Todos los tipos en fact existen en dim_tipo_mantenimiento")

# Verificar que todos los fecha_inicio_id en fact existen en dim_tiempo
fechas_en_fact = set(fact_ordenes['fecha_inicio_id'].dropna().unique())
fechas_en_dim = set(dim_tiempo['fecha_id'].unique())
fechas_huerfanas = fechas_en_fact - fechas_en_dim

if fechas_huerfanas:
    print(f"⚠️  Fechas en fact sin dim: {len(fechas_huerfanas)}")
else:
    print(f"✓ Todas las fechas en fact existen en dim_tiempo")

print(f"\n✓ Integridad referencial verificada")

## 14. Guardar Modelo Dimensional en Silver/

In [ ]:
def save_parquet_to_s3(df, bucket, key, descripcion):
    """Guarda un DataFrame en S3 como Parquet."""
    temp_file = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
    temp_file.close()
    
    df.to_parquet(temp_file.name, index=False)
    s3_client.upload_file(temp_file.name, bucket, key)
    
    Path(temp_file.name).unlink()
    print(f"✓ {descripcion} → s3://{bucket}/{key} ({len(df):,} registros)")

print("=== Guardando Modelo Dimensional en S3 ===")

# Guardar fact_ordenes
save_parquet_to_s3(
    fact_ordenes,
    BUCKET_NAME,
    f"{SILVER_PREFIX}/fact_ordenes_{FECHA_PROCESO}.parquet",
    "fact_ordenes"
)

# Guardar dim_tiempo
save_parquet_to_s3(
    dim_tiempo,
    BUCKET_NAME,
    f"{SILVER_PREFIX}/dim_tiempo_{FECHA_PROCESO}.parquet",
    "dim_tiempo"
)

# Guardar dim_tipo_mantenimiento
save_parquet_to_s3(
    dim_tipo_mant,
    BUCKET_NAME,
    f"{SILVER_PREFIX}/dim_tipo_mantenimiento_{FECHA_PROCESO}.parquet",
    "dim_tipo_mantenimiento"
)

# Guardar dim_activos
save_parquet_to_s3(
    dim_activos,
    BUCKET_NAME,
    f"{SILVER_PREFIX}/dim_activos_{FECHA_PROCESO}.parquet",
    "dim_activos"
)

## 15. Resumen

In [ ]:
print("=" * 60)
print("RESUMEN - Bronze → Silver")
print("=" * 60)
print(f"\n📥 ENTRADA")
print(f"  - Fuente: raw/obra-lineal/")
print(f"  - Registros iniciales: {registros_iniciales:,}")

print(f"\n🧹 LIMPIEZA")
print(f"  - Duplicados eliminados: {duplicados:,}")
print(f"  - Horas inválidas: {invalidos_horas:,}")
print(f"  - Fechas inválidas: {invalidos_fechas:,}")
print(f"  - Sin equipo_id: {sin_equipo:,}")
print(f"  - Tipos inválidos: {tipos_invalidos:,}")
print(f"  - Registros finales: {len(ordenes):,}")

print(f"\n📊 MODELO DIMENSIONAL")
print(f"  - fact_ordenes: {len(fact_ordenes):,} registros")
print(f"  - dim_tiempo: {len(dim_tiempo):,} registros")
print(f"  - dim_tipo_mantenimiento: {len(dim_tipo_mant):,} registros")
print(f"  - dim_activos: {len(dim_activos):,} registros")

print(f"\n✓ Integridad referencial: OK")
print(f"✓ Modelo dimensional guardado en: silver/obra-lineal/")
print("=" * 60)
print("\n➡️  Siguiente paso: Ejecutar notebook 07.3 (Silver → Gold)")